In [1]:
import time
import requests
import grpc
import statistics
import prediction_pb2
import prediction_pb2_grpc

# === Configuraciones ===
FASTAPI_URL = "http://localhost:8800/predict/"
GRAPHQL_URL = "http://localhost:8801/graphql"
GRPC_HOST = "localhost:50051"
N_RUNS = 10

In [2]:
def benchmark_rest():
    sample_rain_input = {
        "features": {
            "Date": "2024-01-01",
            "Location": "Albury",
            "MinTemp": 13.6,
            "MaxTemp": 28.3,
            "Rainfall": 2.6,
            "Evaporation": 3.5,
            "Sunshine": 5.2,
            "WindGustDir": "WNW",
            "WindGustSpeed": 44.3,
            "WindDir9am": "WNW",
            "WindDir3pm": "W",
            "WindSpeed9am": 41.9,
            "WindSpeed3pm": 43.5,
            "Humidity9am": 68.6,
            "Humidity3pm": 81.3,
            "Pressure9am": 1008,
            "Pressure3pm": 1007.6,
            "Cloud9am": 6,
            "Cloud3pm": 8,
            "Temp9am": 16.7,
            "Temp3pm": 25.6,
            "RainToday": False
        }
    }

    sample_not_rain_input = {
        "features": {
            "Date": "2024-01-01",
            "Location": "Albury",
            "MinTemp": 13.6,
            "MaxTemp": 14.0,
            "Rainfall": 0,
            "Evaporation": 0,
            "Sunshine": 5.2,
            "WindGustDir": "WNW",
            "WindGustSpeed": 44.3,
            "WindDir9am": "WNW",
            "WindDir3pm": "W",
            "WindSpeed9am": 41.9,
            "WindSpeed3pm": 43.5,
            "Humidity9am": 30,
            "Humidity3pm": 30,
            "Pressure9am": 1008,
            "Pressure3pm": 1007.6,
            "Cloud9am": 6,
            "Cloud3pm": 8,
            "Temp9am": 16.7,
            "Temp3pm": 25.6,
            "RainToday": False
        }
    }

    times = []
    for _ in range(N_RUNS):
        start = time.time()
        r = requests.post(FASTAPI_URL, json=sample_rain_input)
        elapsed = time.time() - start
        times.append(elapsed)

        assert r.json()['int_output'] == 1  # Asegurarse que predice lluvia

        start = time.time()
        r = requests.post(FASTAPI_URL, json=sample_not_rain_input)
        elapsed = time.time() - start
        times.append(elapsed)

        assert r.json()['int_output'] == 0  # Asegurarse que predice no lluvia

    return statistics.mean(times), statistics.stdev(times)

In [3]:
def benchmark_graphql():
    query_rain = """
    query {
        predict(
            Date: "2024-01-01",
            Location: "Albury",
            MinTemp: 13.6,
            MaxTemp: 28.3,
            Rainfall: 2.6,
            Evaporation: 3.5,
            Sunshine: 5.2,
            WindGustDir: "WNW",
            WindGustSpeed: 44.3,
            WindDir9am: "WNW",
            WindDir3pm: "W",
            WindSpeed9am: 41.9,
            WindSpeed3pm: 43.5,
            Humidity9am: 68.6,
            Humidity3pm: 81.3,
            Pressure9am: 1008,
            Pressure3pm: 1007.6,
            Cloud9am: 6,
            Cloud3pm: 8,
            Temp9am: 16.7,
            Temp3pm: 25.6,
            RainToday: false
        ) {
            intOutput
            strOutput
        }
    }
    """

    query_not_rain = """
    query {
        predict(
            Date: "2024-01-01",
            Location: "Albury",
            MinTemp: 13.6,
            MaxTemp: 14.0,
            Rainfall: 0,
            Evaporation: 0,
            Sunshine: 5.2,
            WindGustDir: "WNW",
            WindGustSpeed: 44.3,
            WindDir9am: "WNW",
            WindDir3pm: "W",
            WindSpeed9am: 41.9,
            WindSpeed3pm: 43.5,
            Humidity9am: 30,
            Humidity3pm: 30,
            Pressure9am: 1008,
            Pressure3pm: 1007.6,
            Cloud9am: 6,
            Cloud3pm: 8,
            Temp9am: 16.7,
            Temp3pm: 25.6,
            RainToday: false
        ) {
            intOutput
            strOutput
        }
    }
    """
    times = []
    for _ in range(N_RUNS):
        start = time.time()
        r = requests.post(GRAPHQL_URL, json={"query": query_rain})
        elapsed = time.time() - start
        times.append(elapsed)

        assert r.json()['data']['predict']['intOutput'] == 1  # Asegurarse que predice lluvia

        start = time.time()
        r = requests.post(GRAPHQL_URL, json={"query": query_not_rain})
        elapsed = time.time() - start
        times.append(elapsed)

        assert r.json()['data']['predict']['intOutput'] == 0  # Asegurarse que predice no lluvia
    return statistics.mean(times), statistics.stdev(times)


In [4]:
def benchmark_grpc():
    req_rain = prediction_pb2.PredictionRequest(
        date="2024-01-01",
        location="Albury",
        min_temp=13.6,
        max_temp=28.3,
        rainfall=2.6,
        evaporation=3.5,
        sunshine=5.2,
        wind_gust_dir="WNW",
        wind_gust_speed=44.3,
        wind_dir_9am="WNW",
        wind_dir_3pm="W",
        wind_speed_9am=41.9,
        wind_speed_3pm=43.5,
        humidity_9am=68.6,
        humidity_3pm=81.3,
        pressure_9am=1008,
        pressure_3pm=1007.6,
        cloud_9am=6,
        cloud_3pm=8,
        temp_9am=16.7,
        temp_3pm=25.6,
        rain_today=False
    )

    req_not_rain = prediction_pb2.PredictionRequest(
        date="2024-01-01",
        location="Albury",
        min_temp=13.6,
        max_temp=14.0,
        rainfall=0,
        evaporation=0,
        sunshine=5.2,
        wind_gust_dir="WNW",
        wind_gust_speed=44.3,
        wind_dir_9am="WNW",
        wind_dir_3pm="W",
        wind_speed_9am=41.9,
        wind_speed_3pm=43.5,
        humidity_9am=30,
        humidity_3pm=30,
        pressure_9am=1008,
        pressure_3pm=1007.6,
        cloud_9am=6,
        cloud_3pm=8,
        temp_9am=16.7,
        temp_3pm=25.6,
        rain_today=False
    )

    times = []
    with grpc.insecure_channel(GRPC_HOST) as channel:
        stub = prediction_pb2_grpc.PredictionServiceStub(channel)
        for _ in range(N_RUNS):
            start = time.time()
            r = stub.Predict(req_rain)
            elapsed = time.time() - start
            times.append(elapsed)

            assert r.int_output == 1  # Asegurarse que predice lluvia

            start = time.time()
            r = stub.Predict(req_not_rain)
            elapsed = time.time() - start
            times.append(elapsed)

            assert r.int_output == 0  # Asegurarse que predice no lluvia
    return statistics.mean(times), statistics.stdev(times)

In [5]:
rest_avg, rest_std = benchmark_rest()
gql_avg, gql_std = benchmark_graphql()
grpc_avg, grpc_std = benchmark_grpc()

print(f"REST    -> {rest_avg:.4f}s ± {rest_std:.4f}")
print(f"GraphQL -> {gql_avg:.4f}s ± {gql_std:.4f}")
print(f"gRPC    -> {grpc_avg:.4f}s ± {grpc_std:.4f}")

REST    -> 0.1285s ± 0.0433
GraphQL -> 0.1051s ± 0.0354
gRPC    -> 0.0402s ± 0.0259
